# Population Dynamics — Lotka-Volterra Simulations

This notebook walks through the full analysis:

1. **Forward Euler from scratch** — implementation and stability analysis
2. **2-species model** — tomato plants vs. aphids
3. **Solver comparison** — Forward Euler vs. adaptive RK45
4. **3-species model** — introducing ladybugs as biological control
5. **Key result** — quantifying the improvement in plant yield

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from models    import two_species, three_species
from solvers   import forward_euler, rk45
from visualize import time_series, phase_portrait, euler_stability, three_species_panel

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## 1. Forward Euler — Implementation & Stability

We implement Forward Euler from scratch:

$$y_{n+1} = y_n + h \cdot f(t_n, y_n)$$

and test it on the simple exponential decay $y' = -\gamma y$ (a single aphid population declining naturally).

The stability boundary for this equation is $h < 2/\gamma$. Below we vary $h$ to see what happens near and beyond that limit.

In [ ]:
def single_species(t, y, gamma=0.1):
    return [-gamma * y[0]]

T_exp    = 60
h_values = [5, 8, 11, 14, 17, 20]

t_analytical = np.linspace(0, T_exp, 300)
y_analytical = 10 * np.exp(-0.1 * t_analytical)

t_list, y_list = [], []
for h in h_values:
    t, y = forward_euler(single_species, [10.0], 0, T_exp, h)
    t_list.append(t)
    y_list.append(y[:, 0])

euler_stability(
    h_values, t_list, y_list,
    t_analytical, y_analytical,
    title='Forward Euler Stability — Aphid Decay (varying step size h)',
)

---
## 2. Two-Species Model: Tomato Plants vs. Aphids

We apply the Lotka-Volterra system to model a tomato crop being damaged by aphids:

$$\frac{dx}{dt} = \alpha x - \beta x y \qquad \frac{dy}{dt} = \delta x y - \gamma y$$

| Parameter | Meaning | Value |
|-----------|---------|-------|
| $\alpha$ | Plant growth rate | 0.3 |
| $\beta$  | Aphid grazing rate | 0.1 |
| $\delta$ | Aphid reproduction per plant consumed | 0.05 |
| $\gamma$ | Aphid natural death rate | 0.1 |

In [ ]:
PARAMS_2 = dict(alpha=0.3, beta=0.1, delta=0.05, gamma=0.1)

t, y, sol = rk45(two_species, [10.0, 10.0], 0, 60, **PARAMS_2)

time_series(
    t, [y[:, 0], y[:, 1]],
    labels=['Tomato plants', 'Aphids'],
    colors=['#2e8b57', '#e07b39'],
    title='2-Species Lotka-Volterra: Tomato Plants vs. Aphids',
)

### Phase Portrait

The phase portrait shows the cyclic orbit in state space — as aphids rise, plants fall, which then causes aphids to fall, allowing plants to recover.

In [ ]:
phase_portrait(
    y[:, 0], y[:, 1],
    xlabel='Plant population', ylabel='Aphid population',
    title='Phase Portrait — Plants vs. Aphids',
)

---
## 3. Solver Comparison: Forward Euler vs. RK45

We compare our hand-written Forward Euler ($h=0.05$) against scipy's adaptive RK45.

RK45 automatically adjusts its step size to maintain a target error tolerance, achieving higher accuracy with far fewer function evaluations.

In [ ]:
t_euler, y_euler = forward_euler(two_species, [10.0, 10.0], 0, 60, h=0.05, **PARAMS_2)
t_rk,   y_rk, _  = rk45(two_species, [10.0, 10.0], 0, 60, **PARAMS_2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
titles = ['Tomato Plants', 'Aphids']
ce = ['#2e8b57', '#e07b39']
cr = ['#145a32', '#784212']
for i, (ax, title) in enumerate(zip(axes, titles)):
    ax.plot(t_euler, y_euler[:, i], color=ce[i], alpha=0.7, lw=1.5, label='Forward Euler (h=0.05)')
    ax.plot(t_rk,   y_rk[:, i],   color=cr[i], lw=2, ls='--',    label='RK45 (adaptive)')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Time (days)')
    ax.set_ylabel('Population'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.suptitle('Forward Euler vs. RK45', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

print(f'RK45 used {len(t_rk)} time points vs {len(t_euler)} for Euler (h=0.05)')

---
## 4. Three-Species Model: Introducing Ladybugs

We extend the model to include ladybugs ($w$) as a biological control agent:

$$\frac{dw}{dt} = \eta\, y\, w - \zeta\, w$$

Aphids now also appear in the ladybug equation, and the aphid equation gains a loss term $-\eta y w$ from predation.

We measure plant yield as $S = \int_0^{60} x(t)\, dt$ — a higher $S$ means more plant biomass survived.

In [ ]:
PARAMS_3 = dict(alpha=0.3, beta=0.1, delta=0.05, gamma=0.3, eta=0.1, zeta=0.5)

t3, y3, sol3 = rk45(three_species, [10.0, 10.0, 10.0], 0, 60, **PARAMS_3)
t2, y2, sol2 = rk45(two_species,   [10.0, 10.0],       0, 60, **PARAMS_2)

t_dense   = np.linspace(0, 60, 2000)
S_with    = np.trapezoid(sol3(t_dense)[0], t_dense)
S_without = np.trapezoid(sol2(t_dense)[0], t_dense)

three_species_panel(
    t3, y3[:, 0], y3[:, 1], y3[:, 2],
    t2, y2[:, 0], y2[:, 1],
    S_with, S_without,
)

print(f'Plant yield without ladybugs : {S_without:.1f}')
print(f'Plant yield with ladybugs    : {S_with:.1f}')
print(f'Improvement                  : +{((S_with - S_without) / S_without * 100):.1f}%')

---
## Summary

| | Without Ladybugs | With Ladybugs |
|---|---|---|
| **Plant yield S** | 111.8 | 442.4 |
| **Improvement** | — | **+295.7%** |

Ladybugs suppress the aphid population before it can devastate the plants, leading to a nearly 4x improvement in total yield over the 60-day growing season.

**What's next:**
- Sensitivity analysis: which parameter has the biggest effect on S?
- Stochastic version: add demographic noise to the ODE system
- Spatial extension: reaction-diffusion PDE on a 2D field